# X-Ray Diffraction (XRD) and Crystal Structure

**Objective:** This lesson introduces X-Ray Diffraction (XRD), the most important technique for determining the atomic and molecular structure of a crystalline material. We will learn the theory behind Bragg's Law, how to interpret a diffraction pattern, and how to use it to calculate the fundamental lattice spacings of a crystal.

**Learning Goals:**
1.  Understand the concept of a crystal lattice and lattice planes.
2.  State and apply **Bragg's Law** to relate diffraction angle to lattice spacing.
3.  Learn to read and interpret a powder XRD pattern (Intensity vs. 2θ).
4.  Use `scipy.signal.find_peaks` to programmatically identify diffraction peaks in a dataset.
5.  Calculate the d-spacings for a known crystal structure (NaCl) and compare them to the expected values.

## Part 1: The Theory - How XRD Works

Crystalline solids are defined by a highly ordered, repeating arrangement of atoms that form a **crystal lattice**. Within this lattice, we can imagine sets of parallel planes that pass through the atoms. 

When a beam of X-rays hits the crystal, these planes act like a series of semi-transparent mirrors. At most angles, the reflected X-rays interfere destructively and cancel each other out. However, at certain specific angles, the reflected waves interfere **constructively**, leading to a strong signal (a peak). 

This condition for constructive interference is described by **Bragg's Law**:
$$ n \lambda = 2d \sin(\theta) $$
where:
*   $n$ is an integer, the order of diffraction (we will assume $n=1$ for simplicity).
*   $\lambda$ is the wavelength of the X-ray source.
*   $d$ is the **d-spacing**, the perpendicular distance between two adjacent lattice planes.
*   $\theta$ is the angle of incidence of the X-ray beam.

An XRD machine scans through a range of angles (reported as $2\theta$) and records the intensity of the diffracted X-rays. The resulting plot is a unique fingerprint of the material's crystal structure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# --- Part 2: Experimental Data ---
# We will use a simulated powder XRD pattern for Sodium Chloride (NaCl).
# The X-ray source is Copper K-alpha, which has a specific wavelength.
lambda_xray = 1.5406 # Angstroms

# Normally, you would load this data from a text file.
# Here we define it directly for the lesson.
two_theta = np.array([27.35, 31.70, 45.45, 53.85, 56.45, 66.20, 75.30]) # in degrees
intensity = np.array([30, 100, 60, 5, 20, 15, 8]) # Relative intensity

# To make a realistic-looking plot, we will simulate the full pattern by creating
# Gaussian peaks at the locations specified above.
def gaussian(x, mu, sigma, amplitude):
    return amplitude * np.exp(-((x - mu) / sigma)**2 / 2)

two_theta_full = np.linspace(20, 80, 1000)
intensity_full = np.zeros_like(two_theta_full)
for i in range(len(two_theta)):
    intensity_full += gaussian(two_theta_full, two_theta[i], 0.1, intensity[i])
intensity_full += np.random.normal(0, 0.5, size=intensity_full.shape) # Add noise

print("XRD data has been simulated.")

In [ ]:
# --- Part 3: Peak Finding and Analysis ---

# `scipy.signal.find_peaks` is a powerful tool for this task.
# 'height=5' tells the function to ignore small noisy peaks.
peak_indices, _ = find_peaks(intensity_full, height=5)
found_peaks_2theta = two_theta_full[peak_indices]

# --- Plotting the Pattern and Found Peaks ---
plt.figure(figsize=(12, 6))
plt.plot(two_theta_full, intensity_full, label='XRD Pattern')
plt.plot(found_peaks_2theta, intensity_full[peak_indices], 'rx', markersize=10, label='Found Peaks')
plt.title('Powder XRD Pattern for NaCl', fontsize=16, weight='bold')
plt.xlabel('Diffraction Angle (2$\theta$)', fontsize=12)
plt.ylabel('Intensity (arbitrary units)', fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

print("Identified Peak Positions (2-theta):")
print(np.round(found_peaks_2theta, 2))

## Part 4: Applying Bragg's Law

Now that we have the peak positions ($2\theta$), we can use Bragg's Law to calculate the d-spacing for each set of crystal planes.

Remember to rearrange the equation and convert $2\theta$ to $\theta$ in radians!
$$ d = \frac{\lambda}{2 \sin(\theta)} $$

In [ ]:
# Convert 2-theta from degrees to theta in radians
theta_rad = np.deg2rad(found_peaks_2theta / 2)

# Apply Bragg's Law
d_spacings = lambda_xray / (2 * np.sin(theta_rad))

print("--- Calculated d-spacings ---")
for i in range(len(found_peaks_2theta)):
    print(f"Peak at 2-theta = {found_peaks_2theta[i]:.2f}° corresponds to a d-spacing of {d_spacings[i]:.3f} Å")

# For a Face-Centered Cubic (FCC) lattice like NaCl, the d-spacings are related
# to the lattice parameter 'a' and the Miller indices (h,k,l) of the planes:
# d = a / sqrt(h^2 + k^2 + l^2)
# The first few allowed reflections for FCC are (111), (200), (220), (311), (222), etc.
a_nacl = 5.64 # Known lattice parameter for NaCl in Angstroms
hkl = np.array([[1,1,1], [2,0,0], [2,2,0], [3,1,1], [2,2,2], [4,0,0], [4,2,0]])
d_theoretical = a_nacl / np.sqrt(np.sum(hkl**2, axis=1))

print("\n--- Comparison with Theoretical Values for FCC NaCl ---")
print("Calculated | Theoretical")
for i in range(len(d_spacings)):
    print(f"   {d_spacings[i]:.3f}    |    {d_theoretical[i]:.3f}")

## Student Challenges

1.  **Unknown Material:** Imagine you are given an XRD pattern for an unknown cubic material. You calculate the d-spacings as: `2.165 Å, 1.875 Å, 1.325 Å`. The ratio of `1/d^2` values for different crystal systems follows specific patterns. For a cubic system, $1/d^2 = (h^2+k^2+l^2)/a^2$. The sequence of allowed $(h^2+k^2+l^2)$ values for Simple Cubic is (1, 2, 3, 4, 5, 6), for Body-Centered Cubic is (2, 4, 6, 8, 10), and for Face-Centered Cubic is (3, 4, 8, 11, 12). Can you determine the crystal system of this unknown material?

2.  **Crystallite Size:** The width of the diffraction peaks is related to the size of the crystals in the powder. The **Scherrer equation** relates the peak width ($β$) to the crystallite size ($L$): $L = \frac{K \lambda}{\beta \cos(\theta)}$. Research this equation. Can you estimate the crystallite size of our NaCl sample, assuming the peak width ($β$) at $45.45^{\circ}$ is about $0.2^{\circ}$? (Remember to convert $\beta$ to radians).